In [1]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *
from eval_functions import *

In [2]:
main_directory = 'model_output'
models = find_folders_with_output(main_directory)

save = True

eval_path = f"evaluate/batch1"
os.makedirs(eval_path, exist_ok=True)

### Model Output to nice JSON and Failure 

In [3]:
def process_files(model, save=save):
    input_dir = f"model_output/{model}/output/"
    output_dir = f"model_output/{model}/formatted/"
    failure_dir = f"model_output/{model}/failed/"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)
    files = os.listdir(input_dir)
    len_files = len(files)
    for filename in files:
        if filename.endswith('.json'):
            input_file_path = os.path.join(input_dir, filename)
            output_file_path = os.path.join(output_dir, filename)
            failure_file_path = os.path.join(failure_dir, filename)
            try:
                file = read_json(input_file_path)
                print(f"Processing file: {filename}")
                if save:
                    shutil.copy(input_file_path, output_file_path)
                    #save_json_to_file(file, output_file_path)
            except Exception as e:
                print(f"Error processing file {filename}: {e}")
                if save:
                    shutil.copy(input_file_path, failure_file_path)
    return len_files

In [6]:
for model in models:
    print(model)
    # 1 - Preprocess Files
    num_out_files = process_files(model, save=save)
    print(num_out_files)
    # 2 - Evaluate Files
    p2_label_path = "chia_label/p2"
    ready_path = f"model_output/{model}/ready"
    failed_model_path = f"model_output/{model}/failed_inner"
    p2_model_formatted_path = f"model_output/{model}/formatted"
    
    for path in [ready_path, failed_model_path, p2_model_formatted_path]:
        os.makedirs(path, exist_ok=True)
    
    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
    model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}
    
    common_ncts = set(label_files.keys()).intersection(model_files.keys())
    labels = []
    predictions = []
    success_data = []
    
    for nct in common_ncts:
        try:
            label_data = read_json(label_files[nct])
            model_data = read_json(model_files[nct])
            print(model_data)
            label_structure = extract_logical_structure(label_data)
            model_structure = extract_logical_structure(model_data)
    
            # Try count words
            label_raw_texts = extract_raw_texts(label_data)
            model_raw_texts = extract_raw_texts(model_data)
            
            missing_texts = label_raw_texts - model_raw_texts
            
            label_words = extract_words(label_raw_texts)
            model_words = extract_words(model_raw_texts)
            missing_words = label_words - model_words
            missing_words_pct = len(missing_words) / len(label_words) * 100 if label_words else 0
            print(f"Processing NCT {nct}")
            print(round(missing_words_pct,2), "%")
            print(label_words.issubset(model_words))
            print("Label Text \n")
            print(label_words)
            print("\nModel Text \n")
            print(model_words)
            print()
            
            success_data.append({
                'NCT': nct,
                'label_AND': label_structure.get('AND', 0),
                'label_OR': label_structure.get('OR', 0),
                'label_NOT': label_structure.get('NOT', 0),
                'label_DEPTH': label_structure.get('depth', 0),
                'model_AND': model_structure.get('AND', 0),
                'model_OR': model_structure.get('OR', 0),
                'model_NOT': model_structure.get('NOT', 0),
                'model_DEPTH': model_structure.get('depth', 0),
                'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
                'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
                'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
                'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0,
                'num_out_files': num_out_files,
            })
    
            labels.append(label_structure)
            predictions.append(model_structure)
            save and shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
        except Exception as e:
            #print(f"Error processing NCT {nct}: {e}")
            save and shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))
    
    df_success = pd.DataFrame(success_data).set_index('NCT')
    df_success.to_csv(eval_path+f'/{model}_eval.csv')
    true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
    predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values
    
    metrics = {}
    for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
        y_true = true_values[:, i]
        y_pred = predicted_values[:, i]
    
        diffs = y_pred - y_true
        pct_greater = (diffs > 0).sum() / len(diffs) * 100
        pct_less = (diffs < 0).sum() / len(diffs) * 100
        pct_equal = (diffs == 0).sum() / len(diffs) * 100
    
        metrics[metric] = {
            'accuracy': round(accuracy_score(y_true, y_pred), 3),
            'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'pct_greater': round(pct_greater, 2),
            'pct_less': round(pct_less, 2),
            'pct_equal': round(pct_equal, 2)
            # 'confusion_matrix': confusion_matrix(y_true, y_pred)
        }
        metrics_df = pd.DataFrame(metrics).T
        num_nct_files = len(df_success)
        metrics_df['num_nct_files'] = num_nct_files
        metrics_df['model_name'] = model
        metrics_df['num_files'] = num_out_files
        # Save Metrics to CSV
        metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))    

Llama-3-8B-Instruct_4_shot
Processing file: Llama-3-8B-Instruct_NCT00050349_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00050349_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00061308_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00061308_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00094861_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00094861_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00122070_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00122070_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00182520_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00182520_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00183885_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00183885_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00198913_exc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00198913_inc_4_shot.json
Processing file: Llama-3-8B-Instruct_NCT00235170_exc_4_shot.jso

In [7]:
# NCT02935855_inc

In [8]:
df_success = pd.DataFrame(success_data).set_index('NCT')
df_success.to_csv(eval_path+f'/{model}_eval.csv')

df_success

,label_AND,label_OR,label_NOT,label_DEPTH,model_AND,model_OR,model_NOT,model_DEPTH,diff_AND,diff_OR,diff_NOT,diff_DEPTH,num_out_files
NCT,,,,,,,,,,,,,
NCT00461136_exc,42,38,1,73,2,4,0,7,-1,-1,-1,-1,300
NCT00198913_inc,0,0,0,1,3,0,0,7,1,0,0,1,300
NCT00639795_exc,24,4,0,39,5,16,0,21,-1,1,0,-1,300
NCT00730301_exc,19,14,1,43,1,18,0,39,-1,1,-1,-1,300
NCT00954850_exc,6,3,0,15,5,5,0,9,-1,1,0,-1,300
...,...,...,...,...,...,...,...,...,...,...,...,...,...
NCT00250640_exc,1,1,0,5,2,3,1,9,1,1,1,1,300
NCT00609531_exc,17,10,2,31,1,2,0,5,-1,-1,-1,-1,300
NCT01490034_inc,8,1,5,17,10,0,2,19,1,-1,-1,1,300


In [10]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    diffs = y_pred - y_true
    pct_greater = (diffs > 0).sum() / len(diffs) * 100
    pct_less = (diffs < 0).sum() / len(diffs) * 100
    pct_equal = (diffs == 0).sum() / len(diffs) * 100


    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'pct_greater': round(pct_greater, 2),
        'pct_less': round(pct_less, 2),
        'pct_equal': round(pct_equal, 2)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_out_files'] = num_out_files
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))

print(f"{len(df_success)} Daten mit {model}")
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")

    print(f"  % Greater: {values['pct_greater']}")
    print(f"  % Less: {values['pct_less']}")
    print(f"  % Equal: {values['pct_equal']}")
    print()
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

270 Daten mit Llama-3-8B-Instruct_4_shot
Metrics for AND:
  Accuracy: 0.119
  Precision: 0.104
  Recall: 0.119
  F1 Score: 0.105
  % Greater: 32.22
  % Less: 55.93
  % Equal: 11.85

Metrics for OR:
  Accuracy: 0.274
  Precision: 0.28
  Recall: 0.274
  F1 Score: 0.269
  % Greater: 33.33
  % Less: 39.26
  % Equal: 27.41

Metrics for NOT:
  Accuracy: 0.644
  Precision: 0.544
  Recall: 0.644
  F1 Score: 0.57
  % Greater: 5.19
  % Less: 30.37
  % Equal: 64.44

Metrics for DEPTH:
  Accuracy: 0.159
  Precision: 0.143
  Recall: 0.159
  F1 Score: 0.142
  % Greater: 25.19
  % Less: 58.89
  % Equal: 15.93



In [11]:
metrics_df

,accuracy,precision,recall,f1_score,pct_greater,pct_less,pct_equal,num_out_files,num_nct_files,model_name
AND,0.119,0.104,0.119,0.105,32.22,55.93,11.85,300,270,Llama-3-8B-Instruct_4_shot
OR,0.274,0.280,0.274,0.269,33.33,39.26,27.41,300,270,Llama-3-8B-Instruct_4_shot
NOT,0.644,0.544,0.644,0.570,5.19,30.37,64.44,300,270,Llama-3-8B-Instruct_4_shot
DEPTH,0.159,0.143,0.159,0.142,25.19,58.89,15.93,300,270,Llama-3-8B-Instruct_4_shot


In [12]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows

,label_AND,model_AND
NCT,,
NCT00440245_exc,0,0
NCT01088750_inc,2,2
NCT01581749_exc,4,4
NCT01032109_exc,4,4
NCT00886158_exc,0,0
NCT01051414_inc,1,1
NCT01346436_inc,2,2
NCT00502567_inc,3,3
NCT01312012_exc,5,5
